<a href="https://colab.research.google.com/github/speediedan/interpretune/blob/main/src/it_examples/notebooks/publish/attribution_analysis/attribution_analysis.ipynb">
  <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab" />
</a>

# Attribution Analysis with Interpretune, Circuit Tracer, and Neuronpedia

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/speediedan/interpretune/blob/main/src/it_examples/notebooks/publish/attribution_analysis/attribution_analysis.ipynb)

## Analyzing Attribution Graphs with Interpretune, Circuit Tracer and Neuronpedia

### Intro

[Interpretune](https://github.com/speediedan/interpretune) is a flexible framework for exploring, analyzing and tuning 
llm world models. In this tutorial, we'll walk through a simple example of using Interpretune to pursue interpretability 
research with Circuit Tracer. As we'll see, Interpretune handles the required execution context composition, allowing us to 
use the same code in a variety of contexts, depending upon the level of abstraction required.

As a long-time PyTorch and PyTorch Lightning contributor, I've found the PyTorch Lightning framework is the right level 
of abstraction for a large variety of ML research contexts, but some contexts benefit from using core PyTorch directly. 
Additionally, some users may prefer to use the core PyTorch framework directly for a wide variety of reasons including 
maximizing portability. As will be demonstrated here, Interpretune maximizes flexibility and portability by adhering to 
a well-defined protocol that allows auto-composition of our research module with the adapters required for execution in 
a wide variety of contexts. In this example, we'll be executing the same module with core PyTorch and PyTorch Lightning, 
demonstrating the use of `Circuit Tracer` w/ Interpretune for circuit discovery and interpretability research.

> Note - **this is a WIP**, but this is the core idea. If you have any feedback, please let me know!

### Setup

#### Notebook Parameters


In [2]:
# Parameters - These will be injected by papermill during parameterized test runs
use_baseline_salient_logits = True  # logits computation mode: True->salient logits, False->specific logits
use_baseline_transcoder_arch = True  # transcoder architecture: True->SingleLayerTranscoder, False->CrossLayerTranscoder
enable_analysis_injection = True  # Toggle analysis injection for analyzing attribution flow
core_log_dir = None  # Directory to save analysis logs (if None, a temp directory will be created)
analysis_config_path = "analysis_injection_config.yaml"  # Base YAML config merged with notebook overrides

#### Interpretune Import

In [3]:
import interpretune as it  # registered analysis ops will be available as it.<op> when analysis is imported

In [4]:
# Import circuit tracer and required modules
from pprint import pformat
from datetime import datetime

from it_examples.seeds import ensure_local_seeds
from interpretune import ITSessionConfig, ITSession
from interpretune.base.call import it_init

~/repos/interpretune/.claude/worktrees/ct-791-909-pin/src/interpretune/analysis/ops/dispatcher.py:1001: Alias 'ct_feature_intervention' already has an assigned op or alias so the alias specified by 'feature_intervention_forward' cannot be mapped to it
~/repos/interpretune/.claude/worktrees/ct-791-909-pin/src/interpretune/analysis/ops/dispatcher.py:1001: Alias 'semantic_direction' already has an assigned op or alias so the alias specified by 'concept_direction' cannot be mapped to it
~/repos/interpretune/.claude/worktrees/ct-791-909-pin/src/interpretune/analysis/ops/dispatcher.py:1001: Alias 'model_forward' already has an assigned op or alias so the alias specified by 'model_fwd' cannot be mapped to it
~/repos/interpretune/.claude/worktrees/ct-791-909-pin/src/interpretune/analysis/ops/dispatcher.py:1001: Alias 'jacobian_lens_read' already has an assigned op or alias so the alias specified by 'jlens_read' cannot be mapped to it
~/repos/interpretune/.claude/worktrees/ct-791-909-pin/src/in

### Enable Analysis Injection

Interpretune includes a lightweight in-notebook analysis injection utility that can temporarily instrument adapter packages (for this notebook, `circuit_tracer`) with small, configurable hooks named analysis points.

This tooling is intentionally experimental and most useful for short-lived, exploratory work: it lets you quickly inspect intermediate values, add ad-hoc diagnostics and exposition (as in this notebook), and prototype ideas without changing the adapter's source tree.

By default, we set `enable_analysis_injection = True` in our parameters cell above to enable the runtime patching. The orchestrator validates hooks, applies them to the target package path, and registers the analysis point functions you provide (whether sourced from files or notebook cells).

<details>
<summary>More on Analysis Injection</summary>

Key characteristics:

- Ephemeral: patches are applied in-process for the notebook session only and do not persist to source files.
- Configuration-driven: hooks are registered via YAML configs plus analysis function mappings that can live in this notebook, external files, or any mix of the two.
- Flexible composition: you can load the base YAML and analysis hook functions from external files, then override or extend them inline in subsequent notebook cells.
- Notebook-friendly output: each analysis point's collected data is available via various helper methods on the `orchestrator` (e.g. `get_analysis_data`, `get_output`), allowing later cells to display the captured values without re-running the instrumentation (as is done in the attribution analysis exposition of this notebook). Analysis events are also written to the configured log (e.g. `/tmp/attribution_flow_analysis_<timestamp>.log`) and optionally to the console when enabled.

The `analysis_injection` utility is a powerful exploratory and expository tool, but for ongoing or production workflows you should upstream a proper hook interface in the relevant adapter package.
</details>

<details>
<summary>Guidance on Analysis Injection Usage</summary>

If you or other users find a recurring need to access intermediate analysis variables, the more robust and maintainable solution is to add a small hook interface to the adapter package itself.

Why prefer adapter-level hooks over regex-based patching:
- Stability: explicit APIs are far less brittle than runtime regex patching.
- Maintainability: adapter maintainers can review, test, and document hook APIs so they remain supported across releases.
- Performance and safety: built-in hooks can be designed to avoid unintended side-effects or excessive overhead.

If an adapter would benefit from exposing intermediate analysis variables, Interpretune recommends opening an issue or a pull request against the adapter repository proposing a small, well-scoped hook API (describe the use case, example call signatures, and what guarantees callers should expect). This is the recommended path for any capability you expect to use repeatedly or across teams.

When to use which approach:
- Use `analysis_injection` for short experiments, expository notebooks like this one, ad-hoc debugging, or rapid iteration on ideas.
- Propose adapter hooks (issue/PR) when you want a repeatable, supported, and long-term inspection facility.

For more details and guidance on safe usage patterns, see the Interpretune project documentation and repo: https://github.com/speediedan/interpretune

</details>


In [5]:
from pathlib import Path
from typing import Any, Dict

import torch

from interpretune.analysis.injection.analysis_hook_patcher import HOOK_REGISTRY
from interpretune.analysis.injection.orchestrator import analysis_log_point, sample_tensor_output
from it_examples.utils.example_helpers import collect_shapes, VarAnnotate
from it_examples.utils.graph_analysis_helpers import plot_ridgeline_convergence
from IPython.display import display

NOTEBOOK_DIR = Path.cwd()
raw_config_path = Path(analysis_config_path)
base_config_path = raw_config_path if raw_config_path.is_absolute() else NOTEBOOK_DIR / raw_config_path
if not base_config_path.exists():
    raise FileNotFoundError(f"Expected config file at {base_config_path}")
base_config_path = base_config_path.resolve()

print(f"Using base analysis injection config at {base_config_path}")

Using base analysis injection config at ~/repos/interpretune/.claude/worktrees/ct-791-909-pin/src/it_examples/notebooks/publish/attribution_analysis/analysis_injection_config.yaml


#### Define/Customize Analysis Points

In [6]:
# Notebook override: add the attribution setup analysis point locally while still
# reusing the module-defined defaults. This demonstrates how to extend the
# external analysis_points module from within the notebook.
def ap_setup_attribution_end(local_vars: Dict[str, Any]) -> None:
    data: Dict[str, Any] = {}
    # Collect shapes from ctx attributes with descriptions
    collect_shapes(
        data,
        local_vars,
        [
            VarAnnotate("ctx.activation_matrix", "n_layers, n_pos, d_transcoder"),
            VarAnnotate("ctx.decoder_vecs", "num_active_features, d_model"),
            VarAnnotate("ctx.encoder_vecs", "num_active_features, d_model"),
            VarAnnotate("ctx.logits", "n_examples (usually 1), n_pos, d_vocab"),
            VarAnnotate("ctx.token_vectors", "n_pos, d_model"),
            VarAnnotate("ctx.error_vectors", "n_layers, n_pos, d_model"),
            VarAnnotate("ctx.encoder_to_decoder_map", "num_active_features"),
            VarAnnotate(
                "ctx.decoder_locations",
                "dims activation_matrix, num_active_features (sparse indices into activation_matrix)",
            ),
        ],
    )
    ctx = local_vars.get("ctx")
    # Add non-shape attributes
    data["n_layers"] = getattr(ctx, "n_layers", None)
    data["_row_size"] = VarAnnotate(
        "ctx._row_size", ctx._row_size, "total_active_feats + error_nodes ((n_layers + 1) * n_pos)  # + logits later"
    )
    analysis_log_point("AttributionContext summary after precomputing activations and vectors", data)


NOTEBOOK_ANALYSIS_FUNCTIONS = {"ap_setup_attribution_end": ap_setup_attribution_end}
print("Registered notebook override analysis point: ap_setup_attribution_end")

Registered notebook override analysis point: ap_setup_attribution_end


#### Define/Customize Analysis Injection Config

In [7]:
# We can customize the base config (`analysis_injection_config.yaml` in this case) with overrides to tweak settings and
# manipulate hook definitions declaratively.
import tempfile
from pathlib import Path

# Ensure the log directory is reachable for this session.
target_log_dir = Path(core_log_dir).expanduser() if core_log_dir else Path(tempfile.gettempdir())
target_log_dir.mkdir(parents=True, exist_ok=True)

# Demonstrate notebook-based overrides: add (or replace) the
# `ap_setup_attribution_end` hook definition directly via config overrides.
analysis_config_overrides = f"""
settings:
  log_dir: {target_log_dir.as_posix()}

file_hooks:
  ap_setup_attribution_end:
    file_path: attribution/attribute_nnsight.py
    enable: true
    regex_pattern: '^\\s*ctx\\s*=\\s*model\\.setup_attribution'
    insert_after: true
    description: "AttributionContext summary at end of phase 0 (added from notebook)"
"""

print("Configured log directory:", target_log_dir.as_posix())
print("Prepared analysis injection config overrides:")
print(analysis_config_overrides)

Configured log directory: /tmp
Prepared analysis injection config overrides:

settings:
  log_dir: /tmp

file_hooks:
  ap_setup_attribution_end:
    file_path: attribution/attribute_nnsight.py
    enable: true
    regex_pattern: '^\s*ctx\s*=\s*model\.setup_attribution'
    insert_after: true
    description: "AttributionContext summary at end of phase 0 (added from notebook)"



#### Instantiate Our Analysis Injector

In [8]:
# Analysis Injection — Setup
# The orchestrator loads the configured analysis point module automatically. We only pass the
# additional notebook-defined hooks via `analysis_functions`.
# Centralized setup: optional env_path (default None), uses the base config from NOTEBOOK_DIR.
from it_examples.utils.example_helpers import required_os_env

# Optional: the user can set `env_path` to a specific .env file path before running this cell.
# If left as None, load_dotenv() will be called without a path so it can auto-discover the .env file.
env_path: str | None = None  # set to '/full/path/to/.env' to override

if enable_analysis_injection:
    # Load environment variables. If env_path is provided, use it; otherwise let load_dotenv auto-discover.
    os_env_reqs = None
    assert required_os_env(env_path=env_path, env_reqs=os_env_reqs)

    # Import orchestrator from the analysis_injection package
    from interpretune.analysis.injection import orchestrator

    print("Setting up analysis injection using base config:", base_config_path)

    # Create the orchestrator which performs the patching. The simplified API handles validation,
    # module loading, and registration automatically.
    analysis_injector = orchestrator.setup_analysis_injection(
        config_path=base_config_path,
        target_package="circuit_tracer",
        config_overrides=analysis_config_overrides,
        analysis_functions=NOTEBOOK_ANALYSIS_FUNCTIONS,
    )

    print("Analysis injection ready. Active patched modules:")
    if analysis_injector.patched_modules:
        for module_name in analysis_injector.patched_modules.keys():
            print(f"  - {module_name}")
    else:
        print("  (No modules patched; check configuration)")

    print("Hook registry status:")
    print(f"  Enabled: {orchestrator.HOOK_REGISTRY._enabled}")
    print(f"  Registered hooks: {len(orchestrator.HOOK_REGISTRY._hooks)}")

    print("\nYou can inspect collected analysis data via orchestrator.get_analysis_data().")
else:
    print("Analysis injection disabled via parameters; skipping setup.")

Setting up analysis injection using base config: ~/repos/interpretune/.claude/worktrees/ct-791-909-pin/src/it_examples/notebooks/publish/attribution_analysis/analysis_injection_config.yaml


~/repos/interpretune/.claude/worktrees/ct-791-909-pin/src/interpretune/analysis/injection/analysis_hook_patcher.py:372: `torch.distributed.reduce_op` is deprecated, please use `torch.distributed.ReduceOp` instead
~/repos/interpretune/.claude/worktrees/ct-791-909-pin/src/interpretune/analysis/injection/analysis_hook_patcher.py:382: `torch.distributed.reduce_op` is deprecated, please use `torch.distributed.ReduceOp` instead
/usr/lib/python3.13/inspect.py:304: FutureWarning: `torch.distributed.reduce_op` is deprecated, please use `torch.distributed.ReduceOp` instead
  return isinstance(object, type)


📝 Analysis output will be logged to: /tmp/attribution_flow_analysis_20261103_142114.log
Analysis injection ready. Active patched modules:
  - circuit_tracer.attribution.attribute_nnsight
  - circuit_tracer.replacement_model.replacement_model_nnsight
  - circuit_tracer.graph
Hook registry status:
  Enabled: True
  Registered hooks: 14

You can inspect collected analysis data via orchestrator.get_analysis_data().


### Configure our IT Session


Here we define or customize our session configuration, which includes:
1. Experiment/task module and datamodule (in this case, 'rte' for the RTE task)
    * We can customize any module, datamodule, or adapter-specific configuration options we want to use. In this case, we set target `circuit_tracer_cfg` that we want to use for our analysis. We also could customize generation parameters, tokenization, the pretrained/config-based model we want to use (in this case, Gemma 2 2B) etc.
2. The adapter context we want to use. In this case, `core` PyTorch (vs e.g. Lightning), `nnsight` as the execution backend circuit-tracer attributes through, and `circuit_tracer` itself.

When an `ITSession` is created, the selected adapter context will trigger composition of the relevant adapters with our experiment/task module and datamodule. The intention of this abstraction is to enable the same experiment/task logic to be used unchanged across a broad variety of PyTorch framework and analytical package contexts.


In [9]:
# Load our demo config via hub-first resolution (cache-backed; seeded from the in-tree publish sources)
ensure_local_seeds()  # idempotent, offline: seed publish sources -> local components cache
base_itdm_cfg, base_it_cfg, dm_cls, m_cls = it.hub.load("speediedan/rte", "rte_demo.gemma2.circuit_tracer+nnsight")
# Optionally override base_it_cfg.core_log_dir with the notebook parameter if provided
if core_log_dir:
    base_it_cfg.core_log_dir = core_log_dir
# If the user requests the baseline salient-logits path, clear the explicit target
# token configuration so the adapter will fall back to the default compute_salient_logits
# implementation. This forces usage of the baseline salient-logits computation.
if use_baseline_salient_logits:
    # Clear any explicit token selection so compute_salient_logits() runs its default path
    base_it_cfg.circuit_tracer_cfg.analysis_target_tokens = None
    base_it_cfg.circuit_tracer_cfg.target_token_ids = None
    print(
        "use_baseline_salient_logits=True: cleared analysis_target_tokens and "
        "target_token_ids -> using default compute_salient_logits path"
    )
else:
    print("use_baseline_salient_logits=False: keeping configured analysis_target_tokens / target_token_ids (if any)")

if enable_analysis_injection:
    try:
        base_it_cfg.circuit_tracer_cfg.verbose = False
        print("✓ Analysis injection enabled: disabled circuit_tracer verbose logging to avoid duplicate logs")
    except Exception as e:
        print(f"⚠️ Could not disable circuit_tracer verbose logging: {e}")


# Configure transcoder architecture selection based on the toggle. When True, use the 'gemma'
# CrossLayerTranscoder (demo). When False, point to the HF SingleLayerTranscoder checkpoint URL.
if use_baseline_transcoder_arch:
    base_it_cfg.circuit_tracer_cfg.transcoder_set = "gemma"
    print(
        "use_baseline_transcoder_arch=True: set transcoder_set='gemma' -> "
        "set transcoder_set to HF URL -> using the SingleLayerTranscoder checkpoint"
    )
else:
    base_it_cfg.circuit_tracer_cfg.transcoder_set = "mntss/clt-gemma-2-2b-426k"
    print(
        "use_baseline_transcoder_arch=False: demo will use CrossLayerTranscoder "
        "instead of the default TranscoderSet of `SingleLayerTranscoder`s"
    )

print(pformat(base_it_cfg.circuit_tracer_cfg))

use_baseline_salient_logits=True: cleared analysis_target_tokens and target_token_ids -> using default compute_salient_logits path
✓ Analysis injection enabled: disabled circuit_tracer verbose logging to avoid duplicate logs
use_baseline_transcoder_arch=True: set transcoder_set='gemma' -> set transcoder_set to HF URL -> using the SingleLayerTranscoder checkpoint
CircuitTracerConfig(backend='nnsight',
                    model_name=None,
                    transcoder_set='gemma',
                    dtype=torch.bfloat16,
                    max_n_logits=10,
                    desired_logit_prob=0.95,
                    batch_size=256,
                    max_feature_nodes=8192,
                    offload='cpu',
                    lazy_encoder=None,
                    lazy_decoder=True,
                    lazy_delta_rank=None,
                    tune_base=False,
                    verbose=False,
                    default_node_threshold=0.8,
                    default_edge_thr

~/repos/interpretune/.claude/worktrees/ct-791-909-pin/src/interpretune/config/shared.py:281: Could not find an auto-composition for <class 'interpretune.adapters.nnsight.config.ITNNsightConfig'> that supports all of the following kwargs: {'circuit_tracer_cfg': CircuitTracerConfig(backend='nnsight', model_name=None, transcoder_set='gemma', dtype=torch.bfloat16, max_n_logits=10, desired_logit_prob=0.95, batch_size=256, max_feature_nodes=8192, offload='cpu', lazy_encoder=None, lazy_decoder=True, lazy_delta_rank=None, tune_base=False, verbose=True, default_node_threshold=0.8, default_edge_threshold=0.98, save_graphs=True, graph_output_dir=None, analysis_target_tokens=['▁Dallas', '▁Austin'], target_token_ids=None, use_neuronpedia=False, intervention_scale_factor=1.0, intervention_max_influence_norm_scale=False, intervention_sign_aware_scale=True, intervention_value=None, intervention_value_source='top_feature_scores', intervention_constrained_layers=None, intervention_freeze_attention=None,

In [10]:
# configure our session with our desired adapter composition: core, nnsight (the execution backend) and circuit_tracer
session_cfg = ITSessionConfig(
    adapter_ctx=(it.Adapter.core, it.Adapter.nnsight, it.Adapter.circuit_tracer),
    datamodule_cfg=base_itdm_cfg,
    module_cfg=base_it_cfg,
    datamodule_cls=dm_cls,
    module_cls=m_cls,
)

# start our session
it_session = ITSession(session_cfg)
print("\nIT Session created successfully!")

# manual init for now
it_init(**it_session)
print("\nIT Session initialized successfully!")

[INFO] interpretune.utils.logging: Loading ReplacementModel with backend: nnsight


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 26 files:   0%|          | 0/26 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/288 [00:00<?, ?it/s]

[INFO] interpretune.utils.logging: NNsight ReplacementModel initialized for Circuit Tracer


~/repos/interpretune/.claude/worktrees/ct-791-909-pin/src/interpretune/adapters/circuit_tracer/adapter.py:215: Lazily loaded transcoder weights (transcoders.0.W_enc, transcoders.0.W_dec, transcoders.1.W_enc and 49 more) are not parameters, so they are left out of this composition's tunable view; set `CircuitTracerConfig.lazy_delta_rank` to tune them through low-rank corrections, or load them eagerly (`lazy_encoder=False`, `lazy_decoder=False`).
[INFO] interpretune.utils.logging: Attempted to clean a key that was not present, continuing without cleaning that key: 'Gemma2Config' object has no attribute 'quantization_config'


... (13 outputs left out)

[INFO] interpretune.utils.logging: Setting up model: InterpretunableModule


[INFO] interpretune.utils.logging: initializing optimizers and schedulers: InterpretunableModule


[INFO] interpretune.utils.logging: Input gradient requirements handled by circuit tracer internally.



IT Session initialized successfully!


### Set Prompts

In [11]:
from tqdm.auto import tqdm

limit_analysis_batches = 1
test_token_limit = -1
force_manual_debug_prompts = True  # Set to True to use manual debug prompts instead of random samples
# specific tokens to analyze, will use tokens associated with top `max_n_logits` if `None`
# analysis_target_tokens: Optional[torch.Tensor] = None

example_prompts = []
ct_module = it_session.module
if not force_manual_debug_prompts:
    dataloader = it_session.datamodule.test_dataloader()
    for epoch_idx in range(1):  # Run for a single epoch for simplicity
        ct_module.current_epoch = epoch_idx
        for batch_idx, batch in tqdm(enumerate(dataloader)):
            if batch_idx >= limit_analysis_batches >= 0:
                break
            # fetch the first test_token_limit from the first example in the batch
            first_ex_in_batch = batch[:1]
            first_ex_in_batch = first_ex_in_batch["input"]
            first_ex_in_batch.squeeze_()
            if test_token_limit > 0:
                first_ex_in_batch = first_ex_in_batch[-test_token_limit:]
            first_ex_in_batch = first_ex_in_batch[first_ex_in_batch != 0]
            example_prompts.append(first_ex_in_batch)
else:
    # Generate attribution graphs for a few example prompts
    example_prompts = [
        # "The capital of France is",
        "The capital of the state containing Dallas is",
        # "When I look at the sky, I see",
    ]

In [12]:
# Set tokenizer context for analysis (if hooks are enabled)
if enable_analysis_injection and analysis_injector:
    from interpretune.analysis.injection.analysis_hook_patcher import HOOK_REGISTRY
    from it_examples.utils.example_helpers import TargetTokenAnalysis

    # Convert target_tokens to IDs using the model's tokenizer
    if analysis_injector.config.shared_context["target_tokens"]:
        target_tokens = analysis_injector.config.shared_context["target_tokens"]
        target_token_analysis = TargetTokenAnalysis(
            tokens=target_tokens, tokenizer=ct_module.model.tokenizer, default_device=ct_module.device
        )

        HOOK_REGISTRY.set_context(
            target_token_ids=target_token_analysis.token_ids, target_token_analysis=target_token_analysis
        )
        print(f"✓ Target tokens set: {target_token_analysis.tokens} → IDs: {target_token_analysis.token_ids}")

✓ Target tokens set: ['▁Dallas', '▁Austin'] → IDs: tensor([26865, 22605], device='cuda:0')


### Generate Basic Attribution Graph
 

In [13]:
print("Generating attribution graphs for example prompts...")
slug_base = "it_circuit_tracer_compute_specific_logits_demo"
results = []

for i, prompt in enumerate(example_prompts):
    print(f"\nProcessing prompt {i + 1}: '{prompt}'")
    slug = f"{slug_base}_{i + 1}_{datetime.now().strftime('%Y%m%d_%H%M%S')}"
    # Process the batch using the session, the adapter will handle tokenization and graph generation
    try:
        graph, local_graph_path, _ = ct_module.generate_graph(prompt=prompt, slug=slug)
        results.append(local_graph_path)
    except Exception as e:
        print(f"  - Error processing prompt: {e}")

print(f"\nProcessed {len(results)} prompts successfully")
# Check and display analysis log file location if available
if enable_analysis_injection and analysis_injector.analysis_log:
    print(f"📝 Analysis log available for inspection: {analysis_injector.analysis_log}")
    print(
        "   The subsequent cells in the `Annotated Attribution Flow Analysis` will display key analysis points values"
        " with any associated annotations in context. You can also inspect the file above for the raw analysis point"
        " values and any additional debug information collected during graph generation."
    )

Generating attribution graphs for example prompts...

Processing prompt 1: 'The capital of the state containing Dallas is'


[INFO] interpretune.utils.logging: Neuronpedia upload not requested. Set upload_to_np to `True` to automatically upload.



Processed 1 prompts successfully
📝 Analysis log available for inspection: /tmp/attribution_flow_analysis_20261103_142114.log
   The subsequent cells in the `Annotated Attribution Flow Analysis` will display key analysis points values with any associated annotations in context. You can also inspect the file above for the raw analysis point values and any additional debug information collected during graph generation.


## Annotated Attribution Flow Analysis

This notebook runs circuit-tracer's **nnsight backend**: the replacement model is an nnsight `LanguageModel` wrapping the Hugging Face `Gemma2ForCausalLM`, and every location attribution reads or writes is an nnsight *envoy* on that model rather than a registered hook point. The walkthrough below follows `circuit_tracer/attribution/attribute_nnsight.py`, `attribution/context_nnsight.py` and `replacement_model/replacement_model_nnsight.py`, which are the modules this notebook's analysis points are injected into.


### Model overview

#### ReplacementModel 

`NNSightReplacementModel` is common to the `TranscoderSet` (set of `SingleLayerTranscoder`s) and `CrossLayerTranscoder` architectures. It adds no modules to the Hugging Face model: it attaches the transcoders as `model.transcoders` and resolves the locations attribution needs **lazily, by pattern**, from circuit-tracer's per-architecture mapping (`utils/tl_nnsight_mapping.py`). Resolving them on demand matters because an envoy such as a function call *inside* a module's forward (`.source`) only exists within a trace.

For `Gemma2ForCausalLM` the mapping resolves to:

| Role in attribution | nnsight location (per `{layer}`) |
| --- | --- |
| Transcoder input, SLT (`ln2.hook_normalized` in TransformerLens terms) | `model.layers[{layer}].pre_feedforward_layernorm.source.self__norm_0`, output |
| Transcoder input, CLT (`hook_resid_mid`) | `model.layers[{layer}].pre_feedforward_layernorm`, input |
| Transcoder output (`hook_mlp_out`) | `model.layers[{layer}].post_feedforward_layernorm`, output |
| Attention patterns (frozen) | `model.layers[{layer}].self_attn.source.attention_interface_0.source.nn_functional_dropout_0` |
| Normalization denominators (frozen) | the `torch_rsqrt_0` call inside `input_layernorm`, `post_attention_layernorm`, `pre_feedforward_layernorm`, `post_feedforward_layernorm`, and the final `model.norm` |
| Pre-logit residual (logit attribution reads from here) | `model`, whose `last_hidden_state` is already final-normed |
| Token embeddings | `model.embed_tokens` (weights `model.embed_tokens.weight`) |
| Unembedding | `lm_head.weight`, exposed as `model.unembed_weight` |

`EnvoyWrapper` hides the input/output distinction, so `feature_input_loc.output` is always the transcoder's input activation whichever side of the module it is read from.


#### Transcoders

- When using a `TranscoderSet` set of `SingleLayerTranscoders` as the transcoder architecture:

  <details>
    <summary>TranscoderSet</summary>

    ```python
        (transcoders): TranscoderSet(
          (transcoders): ModuleList(
            (0): SingleLayerTranscoder(
              (activation_function): JumpReLU(
                threshold=Parameter containing:
                tensor(0.5664, device='cuda:0', dtype=torch.bfloat16), bandwidth=0.1
              )
            )
            ...
            (25): SingleLayerTranscoder(
              (activation_function): JumpReLU(
                threshold=Parameter containing:
                tensor(6.1250, device='cuda:0', dtype=torch.bfloat16), bandwidth=0.1
              )
            )
          )
        )
    ```
  </details>
  <br/>
- When using a `CrossLayerTranscoder` as the transcoder architecture:
  ```python
    (transcoders): CrossLayerTranscoder()
  ```

  <details>
    <summary>Note on Lazy Decoders</summary>

    - `lazy_decoders` is by default `True` so to get W_dec (without excessive memory demands) we log W_dec manually from within `compute_attribution_components`
      ```python
      {n:p.shape for n,p in model.transcoders.named_parameters()}
      {'W_enc': torch.Size([26, 16384, 2304]), 'b_dec': torch.Size([26, 2304]), 'b_enc': torch.Size([26, 16384])}
      ```
    - notice since each CLT feature has a single encoder weight but decoder weights that output to each subqeuent layer MLP output, our W_dec are shaped accordingly, `n_output_layers = self.n_layers - layer_id`
    - the W_dec then is shaped `(d_transcoder, n_output_layers, d_model)`
    - in addition to lazily loading decoder weights on demand to save memory, only the needed feature ids are loaded as well
      ```python
      w_dec_shapes = {}
      for l in range(self.n_layers):
          path = os.path.join(self.clt_path, f"W_dec_{l}.safetensors")
          with safe_open(path, framework="pt", device=self.device.type) as f:
              w_dec_shapes[l] = f.get_slice(f"W_dec_{l}")[:].to(device).to(dtype).shape
      print(w_dec_shapes)
      {
      0: torch.Size([16384, 26, 2304]), 
      1: torch.Size([16384, 25, 2304]), 
      ...
      24: torch.Size([16384, 2, 2304]), 
      25: torch.Size([16384, 1, 2304])
      }
      ```
    </details>

### Precompute Activations and Generate `AttributionContext`


#### Activation Precomputation



- This phase precomputes the replacement model's transcoder activations and error vectors, saving them and the token embeddings.
```python
    ctx = model.setup_attribution(input_ids)
```

- `setup_attribution` runs one plain nnsight trace (under `torch.no_grad`) that reads the embedding output and the transcoder input and output at every layer and saves them with the logits; no hooks are registered:

    ```python
        with self.trace(tokens):
            embedded = save(self.embed_location.output)
            mlp_in_cache, mlp_out_cache = [], []
            for feature_input_loc, feature_output_loc in zip(
                self.feature_input_locs, self.feature_output_locs
            ):
                mlp_in_cache.append(feature_input_loc.output)
                y = feature_output_loc.output
                if y.ndim == 2:
                    y = y.unsqueeze(0)
                mlp_out_cache.append(y)

            mlp_in_cache = save(torch.cat(mlp_in_cache, dim=0))
            mlp_out_cache = save(torch.cat(mlp_out_cache, dim=0))
            logits = save(self.output.logits)
    ```
- `save(...)` is what makes a value survive the trace: inside it, `feature_input_loc.output` is a proxy for a tensor the forward pass has not produced yet, and after the trace exits the saved names hold the concrete tensors.
- This pass collects the original model's MLP inputs and outputs without the trained transcoders. Nothing is detached here; freezing attention and normalization, and routing the MLP output through the transcoder skip connection, happen later in the attribution forward pass (Phase 1).


#### Transcoder Architecture-Specific Attribution

- `compute_attribution_components` is called to collect all attribution_data required for `AttributionContext`
- this method is transcoder type specific, single layer transcoders form a `TranscoderSet` that have this method, `CrossLayerTranscoders` have a different version
```python
    attribution_data = self.transcoders.compute_attribution_components(mlp_in_cache, self.zero_positions)
```
- we construct the `activation_matrix` using `compute_attribution_components` which allows multiple different transcoder architectures to be used

- For `TranscoderSet` (set of `SingleLayerTranscoder`)

    <details>
    <summary>TranscoderSet Attribution Context</summary><br/>

    - note that this implementation is for the `SingleLayerTranscoders` not cross-layer transcoders. CLTs have same number of encoder parameters but `num_layers/2 `times more decoder parameters and have separate decoder vectors for each subsequent layer. We can see the current reconstruction uses just the corresponding input to that layer and collects that SLT's output. See the `CrossLayerTranscoder` version of `compute_attribution_components` for CLT mechanics

    - we construct per layer sparse activations (zeroing out bos activations also) using our trained transcoders (remember ReplacementMLP is the original model with specially instrumented MLP hooks for inspection/replacement of activations etc)

    - here is where the reconstruction is calculated by passing in the captured/cached mlp_input (mlp_in_cache) to the relevant trained transcoders 

    - `compute_attribution_components` uses `encode_sparse` and `decode_sparse` methods on each transcoder layer to collect our required attribution context and package it in a `AttributionContext` dataclass:
        - `activation_matrix`: Sparse (n_layers, n_pos, d_transcoder) activations
        - `reconstruction`: (n_layers, n_pos, d_model) reconstructed outputs
        - `encoder_vecs`: Concatenated encoder vectors for active features
        - `decoder_vecs`: Concatenated decoder vectors (scaled by activations)
        - `encoder_to_decoder_map`: Mapping from encoder to decoder indices

        - `encode_sparse`
            - accepts incoming activations (for SLT, feature_input_hook is `ln2.hook_normalized`) and uses the current layer's self.W_enc and b_enc to calc pre_acts and the activation function to calc the acts, sets the bos acts to 0 and calculates the active (nnz, indices()) encoders
            - these are the local replacement model transcoder preactivations
            - gets non-zero indices, getting the trained transcoder encoder layers (transposed) for active features
            - **NOTE**: crucially, these target transcoder feature preactivations are linear in each upstream source transcoder feature activations since we freeze attention patterns and normalization denominators!
            - example sparse_acts, active_encoder shapes for gemma layer 0:
            ```python
            sparse_acts.shape  
            torch.Size([9, 16384])  # n_pos, d_transcoder
            active_encoders.shape
            torch.Size([634, 2304]) # num_active_features for the layer, d_model
            ```
        - `decode_sparse`
            - accepts the decoded output activations and scales the relevant W_dec indices by the activations
            - Return decoder rows for **active** features only from the trained transcoders 
            - uses `indices()` to get non-zero indices so requires sparse tensor
            - for each active feature index for each layer, creates a [n_active_features_for_layer, d_model] tensor
            - example shapes for gemma layer 0
                ```python
                W_dec.shape
                torch.Size([16384, 2304])
                transcoders[layer].W_dec[feat_idx].shape
                torch.Size([634, 2304])
                ```
    </details>
    <br/>

- For `CrossLayerTranscoder` Attribution Context

    <details>
    <summary>CrossLayerTranscoder Attribution Context</summary><br/>
    
    - a crucial difference between SLT and CLT attribution flows is that for SLT, `decode_sparse` returns a per-layer reconstruction, whereas for CLT, a separate `compute_reconstruction` step is required to calculate and store the reconstruction separately for each subsequent layer it outputs to (since each layer has a separate decoder for each subsequent layer)
    - encode_sparse
    ```python
        sparse_acts, active_encoders = transcoder.encode_sparse(mlp_inputs[layer], zero_first_pos=True)
    ```
    - select_decoder_vectors
    ```python
            pos_ids, layer_ids, feat_ids, decoder_vectors, encoder_to_decoder_map = (
                self.select_decoder_vectors(features)
        )
    ```
    - compute_reconstruction
    ```python
        reconstruction = self.compute_reconstruction(pos_ids, layer_ids, decoder_vectors)    
    ```
    </details>

**Analysis Point Data**: See below sampled data for the current attribution example at the end of Transcoder-Specific attribution described above.

In [14]:
analysis_injector.get_output("ap_compute_attribution_end")

Key,Inspected Output,Annotation
attribution_data['reconstruction'].shape,"(26, 9, 2304)","n_layers, n_pos, d_model"
attribution_data['decoder_vecs'].shape,"(7137, 2304)","num_active_features, d_model"


#### Error Vectors and Reconstruction

- Finally, we compute the error vectors: `mlp_out_cache` holds the original MLP outputs and `attribution_data["reconstruction"]` the outputs reconstructed from the transcoder features, so their difference is what the transcoders fail to explain. The positions the transcoders were not trained on (`zero_positions`, the `<bos>` position for this model) are zeroed:
    ```python
    error_vectors = mlp_out_cache - attribution_data["reconstruction"]
    error_vectors[:, self.zero_positions] = 0
    ```
- we also save the token vectors, the embedding location's own output saved in the trace above. That is the tensor whose gradient attribution reads, so each token edge is that gradient times that tensor. A row of the embedding weights would not do here: `Gemma2TextScaledWordEmbedding` multiplies by $\sqrt{d_{\mathrm{model}}}$ inside the embedding module, so a raw row is smaller than what the model actually adds by that factor:
    ```python
    token_vectors = embedded[0].detach().to(attribution_device)  # (n_pos, d_model)
    ```
- all the per-layer active feature decoder vectors are scaled by how much each transcoder feature was activated
- at the end of `setup_attribution` we package the attribution components into an `AttributionContext`




**Analysis Point Data**: See below sampled data for the current attribution example at the end of `Precompute Activations and Generate AttributionContext`.


In [15]:
analysis_injector.get_output("ap_setup_attribution_end")

Key,Inspected Output,Annotation
ctx.activation_matrix.shape,"(26, 9, 16384)",
ctx.decoder_vecs.shape,"(7137, 2304)",
ctx.encoder_vecs.shape,"(7137, 2304)",
ctx.logits.shape,"(1, 9, 256000)",
ctx.token_vectors.shape,"(9, 2304)",
ctx.error_vectors.shape,"(26, 9, 2304)",
ctx.encoder_to_decoder_map.shape,"(7137,)",
ctx.decoder_locations.shape,"(2, 7137)",
n_layers,26,
_row_size,7380,total_active_feats + error_nodes ((n_layers + 1) * n_pos) # + logits later


In [16]:
analysis_injector.get_output("ap_precomputation_phase_end")

Key,Value
target_token_ids,"[26865, 22605]"
target_logits,"[23.25, 25.875]"
target_tokens,"['▁Dallas', '▁Austin']"


### Forward Pass

- The attribution forward pass is a single nnsight trace over the prompt expanded to `batch_size` copies, so with 9 tokens and a `batch_size` of 256 the traced input is `torch.Size([256, 9])`. Everything inside the trace is *declared* rather than hooked: it states what to read, overwrite or detach at each location, and nnsight executes it in forward order.

    ```python
    with model.trace() as tracer:
        with tracer.invoke(input_ids.expand(batch_size, -1)):
            pass

        detach_barrier = tracer.barrier(2)

        model.configure_gradient_flow(tracer)
        model.configure_skip_connection(tracer, barrier=detach_barrier)
        ctx.cache_residual(model, tracer, barrier=detach_barrier)
    ```
- `configure_gradient_flow` sets up the linearization attribution depends on:
    - the embedding output gets `requires_grad = True`, so token nodes receive gradient;
    - every attention pattern and every normalization denominator (the `torch_rsqrt_0` inside each RMSNorm) is replaced by its `.detach()`. **This is what makes each target's pre-activation linear in the upstream feature activations**: with attention patterns and normalization scales frozen, the only paths left are linear ones.
- `configure_skip_connection` rewrites each transcoder output location as `skip + (output - skip).detach()`, where `skip` is the transcoder's skip term (zero for transcoders without one). The forward value is unchanged; the backward pass sees only the skip path through the MLP, so gradient does not flow through the MLP's own nonlinearity.
- `cache_residual` records, for use in the backward passes that follow:
    - `ctx._resid_activations`: the transcoder input at every layer, then the pre-logit `last_hidden_state` as the final entry (already passed through the model's final norm, which is the point logits are attributed from);
    - `ctx._feature_output_activations`: the embedding output, then each layer's transcoder output location.
- `tracer.barrier(2)` synchronizes the two invocations that write and read the same transcoder output location (the skip-connection rewrite and the residual cache), so the cache sees the rewritten value.
- after the forward pass the original MLP modules can be offloaded (`[layer.mlp for layer in model.model.layers]`), since attribution now only needs the cached tensors and the graph built through them.


####  Score Computation During the Backward Pass


- Where a hook-based backend registers backward hooks during the forward pass, the nnsight backend has none: scores are computed **inside the backward pass itself**, in `AttributionContext.compute_batch`, by reading the gradient that has reached each transcoder output location and contracting it with the source vectors written there.
- The same mechanism serves logit attribution (Phase 3) and feature attribution (Phase 4)
    - for phase 3, the gradient injected at the pre-logit residual is the demeaned unembedding vector of each target logit;
    - for phase 4, the gradient injected at a feature's input location is that feature's encoder vector (see `Key Attribution Flow Summary` below)
    - see `compute_batch` below for how one backward pass fills every source row for a batch of targets

- Score Computation Details
    <details>
    <summary>Per-Node-Type Score Computation</summary><br/>

    - feature node scores are computed layerwise:
        1. using active features (non-zero activations) from the precomputed `activation_matrix` sparse indices
            ```python
            nnz_layers, nnz_positions = self.decoder_locations
            ```
            - `nnz_layers, nnz_positions` are the non-zero indices for the layers and positions dimensions of `activation_matrix.indices()`
            - for this gemma example, layer 25 has 265 active features distributed across all 8 non-`<bos>` positions, e.g. 31 features active at position 8 (the counts below are from a representative SLT run; they vary slightly between runs and backends at `bfloat16`):
                ```python
                nnz_layers[-265:].unique()
                tensor([25], device='cuda:0')
                nnz_positions[-265:].unique()
                tensor([1, 2, 3, 4, 5, 6, 7, 8], device='cuda:0')
                nnz_positions[-31:].unique()
                tensor([8], device='cuda:0')
                ```
        2. by contracting the gradient with the activation-scaled decoder vectors of the features writing to that layer
            ```python
            def compute_feature_attributions(self, layer, grads):
                nnz_layers, nnz_positions = self.decoder_locations

                # Feature nodes - use decoder_locations to find decoders that write to this layer
                layer_mask = nnz_layers == layer
                if layer_mask.any():
                    self.compute_score(
                        grads,
                        self.decoder_vecs[layer_mask],
                        write_index=self.encoder_to_decoder_map[layer_mask],
                        read_index=np.s_[:, nnz_positions[layer_mask]],
                    )
            ```
        - The `np.s_` indexing constructs a slice that selects the appropriate gradient vectors from the batch.
            - All rows in the first dimension (the batch dimension; for logit attribution only the first `n_logits` of the 256 rows carry an injected gradient) are selected.
            - For the second dimension, the positions associated with non-zero feature activations in `activation_matrix` for that layer.
        - This selects each target's gradient once per active feature at that feature's position.
        - Concretely, for layer 25 in the gemma SLT example, the 265 active decoder vectors for that layer are each contracted with the gradient at their own position, and the results are written to those features' rows of the score buffer:
            ```python
            # these are the pos indices associated with the active features for layer 25
            read_index[1].shape
            torch.Size([265])

            write_index.shape  # corresponding active feature decoder vec indices
            torch.Size([265])

            grads.shape
            torch.Size([256, 9, 2304])  # batch_size, n_pos, d_model
            grads.to(output_vecs.dtype)[read_index].shape
            torch.Size([256, 265, 2304])  # batch_size, num active position/feature combos for layer, d_model

            output_vecs.shape
            torch.Size([265, 2304])  # num active position/feature combos for layer, d_model
            ```

    - error and token node scores are computed the same way, except:
        1. an `error_offset` function computes the rows they occupy, after all the feature rows
            ```python
            def error_offset(layer: int) -> int:  # starting row for this layer
                return self.activation_matrix._nnz() + layer * n_pos
            ```
        2. the source vectors are the precomputed `error_vectors` and `token_vectors` rather than decoder vectors
            ```python
            def compute_error_attributions(self, layer, grads):
                self.compute_score(
                    grads,
                    self.error_vectors[layer],
                    write_index=np.s_[error_offset(layer) : error_offset(layer + 1)],
                )

            def compute_token_attributions(self, grads):
                tok_start = error_offset(n_layers)
                self.compute_score(
                    grads,
                    self.token_vectors,
                    write_index=np.s_[tok_start : tok_start + n_pos],
                )
            ```

    - all three node types reduce to one contraction, which accumulates the scores for the current batch of targets into a buffer with one row per source node:

        ```python
        def compute_score(
            self,
            grads: torch.Tensor,
            output_vecs: torch.Tensor,
            write_index: slice,
            read_index: slice | np.ndarray = np.s_[:],
        ) -> None:
            proxy = weakref.proxy(self)
            proxy._batch_buffer[write_index] += einsum(
                grads.to(output_vecs.dtype)[read_index],
                output_vecs,
                "batch position d_model, position d_model -> position batch",
            )
        ```
    </details>





**Analysis Point Data**: See below sampled data for the current attribution example at the end of `Forward Pass`.

In [17]:
analysis_injector.get_output("ap_forward_pass_end")

Key,Value
target_token_ids,"[26865, 22605]"
target_logits,"[31.0, 39.25]"
target_tokens,"['▁Dallas', '▁Austin']"


### Build Input Vectors

We filter active features of our cached transcoder activation_matrix and depending on our target logits mode, generate our feature matrices we want to use in subsequent analysis by using either the:

1. configured cumulative probability and `max_n_logits` if in the baseline default `compute_salient_logits` target logits mode
2. specific logits specified by token id or token if we use our `compute_specific_logits` target logits mode

**Analysis Point Data**: See below sampled data for the current attribution example at the end of `Build Input Vectors`.

In [18]:
analysis_injector.get_output("ap_build_input_vectors_end")

Key,Inspected Output,Annotation
logit_idx,"[22605, 573, 235248, 9778, 9447, 235292, 476, 26865, 3151, 109]",
logit_p,"[0.337891, 0.109863, 0.075684, 0.040527, 0.040527, 0.035645, 0.031494, 0.024536, 0.021606, 0.016846]",logit probabilities
target_tokens,"['▁Austin', '▁Dallas']",
target_logit_indices,"[0, 7]",
target_logit_p,"[0.337891, 0.024536]",
logit_cumulative_prob,0.734375,
total_nodes,7390,
max_feature_nodes,7137,
total_active_feats,7137,
n_logits,10,


### Core `compute_batch` Attribution Logic

In the `Compute Logit Attributions` and `Compute Feature Attributions` phases, it's important to note we aren't using conventional gradient propagation but rather using a backward pass as a convenient orchestration mechanism to calculate our desired node attributions via custom gradient injection.

The core mechanics of this are:

1. The `compute_batch` method of `AttributionContext`, shared by the Logit and Feature Node Attribution phases:
    <details>
    <summary>Snapshot of `compute_batch`</summary><br/>

    ```python
        def compute_batch(
            self,
            layers: torch.Tensor,
            positions: torch.Tensor,
            inject_values: torch.Tensor,
            retain_graph: bool = True,
        ) -> torch.Tensor:
            batch_size = self._resid_activations[0].shape[0]
            self._batch_buffer = torch.zeros(
                self._row_size,
                batch_size,
                dtype=inject_values.dtype,
                device=inject_values.device,
            )

            # Custom gradient injection (per-layer registration)
            batch_idx = torch.arange(len(layers), device=layers.device)

            def _inject(grad_point, *, batch_indices, pos_indices, values):
                grads_out = grad_point.grad.clone()
                grads_out.index_put_((batch_indices, pos_indices), values.to(grads_out.dtype))
                grad_point.grad = grads_out

            layers_in_batch = sorted(layers.unique().tolist(), reverse=True)

            last_layer = max(layers_in_batch)
            with self._resid_activations[last_layer].backward(
                gradient=torch.zeros_like(self._resid_activations[last_layer]),
                retain_graph=retain_graph,
            ):
                for layer in reversed(range(last_layer + 1)):
                    if layer != last_layer:
                        grad = self._feature_output_activations[layer + 1].grad.clone()
                        self.compute_feature_attributions(layer, grad)
                        self.compute_error_attributions(layer, grad)

                    mask = layers == layer
                    if mask.any():
                        _inject(
                            grad_point=self._resid_activations[layer],
                            batch_indices=batch_idx[mask],
                            pos_indices=positions[mask],
                            values=inject_values[mask],
                        )

                self.compute_token_attributions(self._feature_output_activations[0].grad)

            buf, self._batch_buffer = self._batch_buffer, None
            return buf.T[: len(layers)]
    ```
    </details><br/>
2. Logit and Feature Attribution-Specific invocations of `compute_batch`
    <details>
    <summary>Logit Attribution Computation</summary><br/>

    ```python
        for i in range(0, len(targets), batch_size):
            batch = targets.logit_vectors[i : i + batch_size]
            rows = ctx.compute_batch(
                layers=torch.full((batch.shape[0],), n_layers),
                positions=torch.full((batch.shape[0],), n_pos - 1),
                inject_values=batch,
            )
            edge_matrix[i : i + batch.shape[0], :logit_offset] = rows.cpu()
            row_to_node_index[i : i + batch.shape[0]] = (
                torch.arange(i, i + batch.shape[0]) + logit_offset
            )
    ```
    </details><br/>

    <details>
    <summary>Feature Attribution Computation</summary><br/>

    ```python
    queue = [pending[i : i + batch_size] for i in range(0, len(pending), batch_size)]

    for idx_batch in queue:
        n_visited += len(idx_batch)

        rows = ctx.compute_batch(
            layers=feat_layers[idx_batch],
            positions=feat_pos[idx_batch],
            inject_values=ctx.encoder_vecs[idx_batch],
            retain_graph=n_visited < actual_max_feature_nodes,
        )

        end = min(st + batch_size, st + rows.shape[0])
        edge_matrix[st:end, :logit_offset] = rows.cpu()
        row_to_node_index[st:end] = idx_batch
        visited[idx_batch] = True
        st = end
        pbar.update(len(idx_batch))
    ```
    </details>


#### Key Attribution Flow Summary

The `compute_score` contraction that `compute_batch` drives, as described below, calculates the product of:
1. the unembedding vectors (for target logit node attribution) or the relevant target layer encoder vectors (for target feature node attribution), carried backward as the gradient
2. and (for feature nodes) the relevant layer's activation-scaled transcoder decoder vectors (which represent the feature space transformation).

**This allows us to score how much each feature (scaled by activation) contributes to the residual stream in the unembedding direction (for logit attribution) or the target transcoder encoder direction (for feature-feature attribution)**


### Compute Logit Attributions

`compute_batch` injects the target vectors as gradient, then walks the layers from the highest one in the batch down to the embeddings, reading the gradient that has arrived at each layer's transcoder output location and scoring every source node that writes there.

In the case of the gemma SLT example with `max_n_logits = 10`, every target is a logit, so all 10 are injected at a single location: the pre-logit residual, the final entry of `ctx._resid_activations` (index 26 for this 26-layer model), at the last position. So `layers_in_batch` = [26]
- concretely:
    ```python
    batch.shape
    torch.Size([10, 2304])
    targets.logit_vectors.shape
    torch.Size([10, 2304])
    (layer, len(inject_values[mask]), batch_idx[mask], positions[mask])
    (26, 10, tensor([0, 1, 2, 3, 4, 5, 6, 7, 8, 9]), tensor([8, 8, 8, 8, 8, 8, 8, 8, 8, 8]))
    ```
- each batch will only have non-zero entries in the first 10 rows of dim 0:
    ```python
    has_nonzero = torch.any(grads.to(output_vecs.dtype)[read_index] != 0, dim=(1, 2))
    nz_indices = torch.where(has_nonzero)[0]
    nz_indices
    tensor([0, 1, 2, 3, 4, 5, 6, 7, 8, 9], device='cuda:0')
    ```
The backward pass is opened as a context on that residual, `with self._resid_activations[26].backward(gradient=zeros, ...)`, and `_inject` overwrites its `.grad` at the target positions with our `logit_vecs` (demeaned unembedding vectors), so the gradient that flows back is exactly the target direction.

As the backward pass runs, each layer's `.grad` at `ctx._feature_output_activations[layer + 1]` is read and contracted with the source vectors that write there, filling the corresponding rows of the score buffer layer by layer through `compute_score`. The number of rows filled per layer corresponds to the per-layer number of active nodes (feature, error or token):
```python
def compute_score(
# ...
    proxy = weakref.proxy(self)
    proxy._batch_buffer[write_index] += einsum(
        grads.to(output_vecs.dtype)[read_index],
        output_vecs,
        "batch position d_model, position d_model -> position batch",
    )
```

Note we `retain_graph` for these backward passes so the same forward graph serves every batch. `compute_batch` then returns the batch buffer (which is simultaneously cleared on the object) holding the feature, error and token node attributions for each of the logits (as `rows`)

The edge matrix is then updated using the calculated node attributions in rows (while leaving the last `max_n_logit` (10) logit entries 0), so the first 10 rows in `edge_matrix` are populated except for the last 10 columns, the remaining elements are all zero at this point since we're just building the logit attribution entries.

We finally update the `row_to_node_index` with the logit node mapping, so in this case, logit rows 0:10 map to the last 10 edge_matrix columns, while the 11th flattened node is the first feature node and doesn't have a mapping yet.


#### Reproducing A Specific Node Attribution Calculation

- Let's inspect the attribution of a specific node (e.g. 6334), to make this concrete. We can set a breakpoint in `compute_score` and validate the calculation written to our buffer for the target index 6334. This is the einsum referenced above:
    ```python
    proxy._batch_buffer[write_index] += einsum(grads.to(output_vecs.dtype)[read_index], output_vecs,
        "batch position d_model, position d_model -> position batch",
    )
    ```
- The relevant shapes of the tensors involved in this calculation are:
    ```python
    grads.to(output_vecs.dtype)[read_index].shape # logit_vecs (demeaned and transposed unembedding vectors)
    torch.Size([256, 81, 2304])  # batch_size, layer 20 active nodes, d_model
    output_vecs.shape  # active decoder_vecs for layer
    torch.Size([81, 2304])       # layer 20 active nodes, d_model
    write_index.shape
    torch.Size([81]) # layer 20 active nodes
    read_index[1].shape
    torch.Size([81]) # layer 20 active nodes
    ```
- In this case, we only run `compute_batch` one time since we have fewer than 256 target nodes we want to calculate source attribution scores for. Since we're using the backward pass for orchestration, a single backward pass scores every layer's source nodes. The write_index writes to our score buffer which for each batch (again, just 1 in this case) is of shape `(n_logits, logit_offset)`
- All feature, error and token node scores are calculated; we show just the feature node scores below. The returned rows for each logit computation batch (just 1) is shape (10, 7358) in this case because we have 10 target logits and 7358 total source nodes (feature, error and token nodes).
- We isolate the relevant indices for our target node 6334, we only have 10 nodes in this batch since we're only calculating source attributions for the 10 target logit nodes.:
    ```python
    target_node_id = 6334
    target_buf_index = torch.where(write_index == target_node_id)
    orig_grad_slice = grads.to(output_vecs.dtype)[read_index].detach()
    grads_slice = orig_grad_slice[:10, target_buf_index, :].squeeze()
    output_vecs_slice = output_vecs[target_buf_index].squeeze()
    (grads_slice.shape, output_vecs_slice.shape)
    (torch.Size([10, 2304]), torch.Size([2304]))
    ```
- We can then reproduce the einsum calculation directly to validate the attribution for our target node 6334
    ```python
    torch.matmul(grads_slice, output_vecs_slice)
    tensor([ 6.5625e+00, -1.4258e-01, -2.7344e-02,  8.3750e+00,  2.7656e+00,
            -2.0020e-01, -6.8359e-03,  6.0312e+00, -2.2339e-02,  2.8931e-02],
        device='cuda:0', dtype=torch.bfloat16)
    ```
- So what are the attributions to our feature node 6334 for our `target_logit_indices`? 

    - We projected our decoder vector (associated with node 6334) into the unembed vec direction for each of our top logits individually.
    - For our target token ('▁Austin', '▁Dallas') logit indexes, elements 0 and 7, we see 6.5625 and 6.0312.
    - As expected, our initial attribution for active feature index 6334 after compute logit attribution:
        - returned buffer:
        ```python
        buf.T[:10][[0,7], 6334]  # returned buffer from `compute_batch`
        tensor([6.5625, 6.0312], device='cuda:0', dtype=torch.bfloat16)
        edge_matrix[[0, 7], 6334]  # logit nodes are initially the first `max_n_logits` target nodes
        tensor([6.5625, 6.0312])
        ```


#### Attribution Node Id to (Graph) Feature Id Mapping



- If you want to associate these internal node attributions to feature ids on our attribution graphs, we need to map the node ids back to feature ids (we have helper functions for this but it's good to understand the mechanics)
- remember the `ctx.activation_matrix` saves non-zero activations by feature_idx (`0:d_transcoder`) for each layer for each position:

```python
        ctx.activation_matrix
        tensor(indices=tensor([[    0,     0,     0,  ...,    25,    25,    25],
                        [    1,     1,     1,  ...,     8,     8,     8],
                        [   41,    96,   253,  ..., 16014, 16302, 16326]]),
        values=tensor([ 2.5156,  3.5312,  0.7070,  ...,  7.4375, 40.7500,
                        12.2500]),
        device='cuda:0', size=(26, 9, 16384), nnz=7115, dtype=torch.bfloat16,
        layout=torch.sparse_coo)
```
- the `encoder_to_decoder_map` map is just a vector `activation_matrix._nnz` long (for SLT) that can be used to associate active encoder and decoder vecs to the activation matrix.
- for example for layer 20, find the positions in our `activation_matrix._nnz` length vector of active decoders
```python
        per_layer_mask = (nnz_layers == 20)
        nnz_positions[per_layer_mask]
        tensor([1, 1, 1, 1, 1, 1, 2, 2, 2, 2, 2, 2, 2, 2, 2, 3, 3, 3, 3, 3, 3, 3, 3, 3,
                3, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5,
                6, 6, 6, 6, 6, 6, 6, 6, 6, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 8, 8, 8,
                8, 8, 8, 8, 8, 8, 8, 8, 8], device='cuda:0')
```
- so our active decoders for layer 20 are

```python
        ctx.decoder_vecs[per_layer_mask].shape
        torch.Size([81, 2304])
```

- and the appropriate activation_matrix indices using encoder_to_decoder_map

```python
        ctx.encoder_to_decoder_map[per_layer_mask]
        tensor([6254, 6255, 6256, 6257, 6258, 6259, 6260, 6261, 6262, 6263, 6264, 6265,
                6266, 6267, 6268, 6269, 6270, 6271, 6272, 6273, 6274, 6275, 6276, 6277,
                6278, 6279, 6280, 6281, 6282, 6283, 6284, 6285, 6286, 6287, 6288, 6289,
                6290, 6291, 6292, 6293, 6294, 6295, 6296, 6297, 6298, 6299, 6300, 6301,
                6302, 6303, 6304, 6305, 6306, 6307, 6308, 6309, 6310, 6311, 6312, 6313,
                6314, 6315, 6316, 6317, 6318, 6319, 6320, 6321, 6322, 6323, 6324, 6325,
                6326, 6327, 6328, 6329, 6330, 6331, 6332, 6333, 6334], device='cuda:0')
```
- for the our example feature we're probing in `ctx.encoder_to_decoder_map`, what is the active feature index that we got our active encoder and decoder vecs from in activation_matrix?

```python
        ctx.activation_matrix.indices().T[6334]
        tensor([   20,     8, 15589], device='cuda:0')
```
- so **layer 20**, **position 8**, **feature id 15589** is the feature that corresponds to node id 6334
- using `activation_matrix` directly to probe all features active at layer 20, position 8 and their corresponding values, we see we can expect feature 15589 to have an activation of 52.0 in our graph UI

```python
        active_indices = ctx.activation_matrix.indices()
        test_mask = (active_indices[0] == 20) & (active_indices[1] == 8)
        filtered_indices = active_indices[:, test_mask]
        filtered_values = ctx.activation_matrix.values()[test_mask]

        filtered_indices.T
        tensor([[   20,     8,   114],
                [   20,     8,   438],
                [   20,     8,  3094],
                [   20,     8,  5433],
                [   20,     8,  5916],
                [   20,     8,  6026],
                [   20,     8, 10118],
                [   20,     8, 10254],
                [   20,     8, 15133],
                [   20,     8, 15276],
                [   20,     8, 15366],
                [   20,     8, 15589]], device='cuda:0')
        filtered_values
        tensor([15.5625,  7.8125, 16.7500,  7.5312, 45.7500, 14.8750, 12.1875, 19.0000,
                8.0000, 14.4375,  7.2812, 52.0000], device='cuda:0',
        dtype=torch.bfloat16)
```

**Analysis Point Data**: See below sampled data for the current attribution example at the end of `Compute Logit Attributions`.

In [19]:
analysis_injector.get_output("ap_compute_logit_attribution_end")

Key,Inspected Output,Annotation
edge_matrix.shape,"(7147, 7390)",
row_to_node_index.shape,"(7147,)",
logit_section.shape,"torch.Size([10, 7137])",
logit_section_zeros_info,"(torch.Size([207]), 6900, 7106)","logit section zeros shape, min node id, max node id"
logit_section_zero_feature_range,"6899: (24, 8, 16258)6900: (25, 1, 209)7106: (25, 7, 15301)7107: (25, 8, 553)","In the final layer, only the final position has non-zero feature attributions."


### Compute Feature Attributions

- Note that the edge_matrix (adjacency matrix) analyzed below is indexed as (target, source) (targets are rows)
- When the graph is pruned, source input edges are normalized for each source node so that they sum to 1 for each target.
- if we are operating on all features (`actual_max_feature_nodes`, the requested `max_feature_nodes` capped at `total_active_feats`, equals `total_active_feats`) we can run `compute_batch` below on all the features
- if we are only operating on a subset of features, we first have to run `compute_partial_influences` to sort all the features by logit influence and then score only those features
- `compute_batch` is run once per batch (256 in this example) of nodes to analyze, similar to logit attribution above but instead of injecting `logit_vecs`, we inject the relevant target feature (transcoder encoder vecs) as the gradient.
- Also, instead of just a single compute_batch (with only 10 of the 256 rows populated) and a single backward pass, we have many more batches to process that will be fully populated (except for the last batch)
    - e.g., for a gemma SLT run with 7115 active features, computing the source influence vectors for every target feature takes 28 batches of 256 nodes each (the last batch has only 203 nodes)

        ```python
        len(queue)
        28
        ```
- Our source vectors are the same as described in logit attribution above (bound in the attribution context during precomputation) but they are contracted with the injected target `encoder_vecs` to calculate the feature-to-feature attributions
- each batch injects its targets at the cached residual of each layer present in the batch (a batch can cross layer boundaries), and the backward pass is opened on the highest of those layers
- as the backward pass walks down from that layer, each lower layer's scaled decoder vectors (or the `token_vectors` at the embeddings) are contracted with the gradient that the injected target `encoder_vecs` produce there
- error nodes are scored the same way, with the `error_vectors` `(n_layers, n_pos, d_model)` playing the role the scaled decoder vectors play for feature nodes
- we retain_graph for as long as we have another batch to compute
- we queue up `update_interval` (default 4) * batch_size (so 1024 in this example) nodes unless we have fewer than that left to process


#### Post-Attribution Edge Matrix

- `edge_matrix` inspection after feature attributions but before reshaping to `full_edge_matrix` (so first 10 rows are still logit nodes)
- at this point we have non-normalized influence scores for all logit/feature feature/feature edges so the target vectors are pretty dense:

```python
        edge_matrix[0, :].count_nonzero()
        tensor(7091)
        edge_matrix[6344, :].count_nonzero()
        tensor(6423)
```
- An error node exists for every token for every layer. We also have the token nodes themselves.
```python
# n_error_nodes = (num_layers + 1) * num_tokens = num_layers * num_tokens + num_tokens
```
- for the gemma SLT example, `edge_matrix` at this point is shaped as (7125, 7368): 
```python
# n_logits(10) + n_feature_nodes(7115) -> 7125
# n_feature_nodes(7115) + n_error_nodes(234) + n_tokens(9) + n_logits(10) -> 7368
```

**Analysis Point Data**: See below sampled data for the current attribution example at the end of `Compute Feature Attributions`.

In [20]:
analysis_injector.get_output("ap_compute_feature_attributions_end")

Key,Inspected Output,Annotation
target_tokens,"['▁Austin', '▁Dallas']",
target_logit_p,"[0.337891, 0.024536]",
top_init_edge_indices,"[6394, 7378, 7361, 6466, 6698][7378, 6394, 6466, 6071, 5957]",
top_init_edge_features,"{6394: (20, 8, 15589), 6466: (21, 8, 5943), 6698: (23, 8, 12237), 7361: 'non-feature node', 7378: 'non-feature node'}{5957: (14, 7, 2268), 6071: (16, 7, 25), 6394: (20, 8, 15589), 6466: (21, 8, 5943), 7378: 'non-feature node'}",
top_init_edge_vals,"[6.5, 6.0625, 3.9375, 3.53125, 2.8125][9.75, 5.96875, 3.90625, 3.125, 2.9375]",
features_processed,7137,
max_features,7137,
progress,100.0%,
edge_matrix.shape,"(7147, 7390)","( (n_logits + n_feature_nodes), n_feature_nodes + n_error_nodes + n_token_nodes + n_logits"


### Graph Packaging

Before packaging the our attribution matrices into a `circuit-tracer` `Graph` object, note our top influencing features are not the same as the raw activations you'll see in the graph from `activation_matrix`. They are our computed feature influences (depending on context, normalized or not-yet-normalized)

In our analysis point below you can see our top inspected token first order and second order feature attributions.

- Note this is not yet converted to abs values and normalized so that the input edges sum to 1. The target logit attribution sums do not correspond directly to logit probs but are rather the non-normalized influence scores
- We refer to the top influencing nodes of the top nodes influencing our inspected logits as per token `pre_prune_2nd_order` matrices.
- Our helper functions mask out the error and token nodes from these 2nd order influence matrices since by definition those nodes won't have input feature nodes.

Prior to graph packaging, we reshape our `edge_matrix` to have the logit nodes at the end and use it to populate the input `full_edge_matrix` which will be passed to the `Graph` constructor as our initial (pre-pruned) adjacency matrix.

Our adjacency matrix will have the following shape (n_total_nodes, n_total_nodes). Where `n_total_nodes` is:
```
n_feature_nodes + n_error_nodes + n_token_nodes + n_logits
```


**Analysis Point Data**: See below sampled data for the current attribution example prior to `Graph Packaging`.

In [21]:
analysis_injector.get_output("ap_graph_creation_start")

Key,Inspected Output,Annotation
full_edge_matrix.shape,"(7390, 7390)",
edge_matrix.shape,"(7147, 7390)",
pre_normalized_logit_node_sum,"[33.336029, 28.37944]",Sum of all source nodes for our target logits before normalization
n_total_nodes,7390,len(selected_features):(n_layers * n_pos):n_pos:n_logits
pre_prune_1st_order_idxs,"'▁Austin': [6394, 7378, 7361, 6466, 6698]'▁Dallas': [7378, 6394, 6466, 6071, 5957]",
pre_prune_1st_order_vals,"'▁Austin': [6.5, 6.0625, 3.9375, 3.5312, 2.8125]'▁Dallas': [9.75, 5.9688, 3.9062, 3.125, 2.9375]",
pre_prune_2nd_order_idxs,"'▁Austin': [7378, 5957, 6071, 2230, 6253][3, 4, 1, 0, 2][3, 4, 1, 0, 2][6251, 6078, 7289, 7307, 6319][6466, 7378, 6394, 6555, 6317]'▁Dallas': [3, 4, 1, 0, 2][7378, 5957, 6071, 2230, 6253][6251, 6078, 7289, 7307, 6319][7378, 5957, 4237, 2230, 7279][7378, 4237, 2230, 7225, 7374]",
pre_prune_2nd_order_vals,"'▁Austin': [50.75, 17.125, 15.125, 10.5625, 10.3125][-inf, -inf, -inf, -inf, -inf][-inf, -inf, -inf, -inf, -inf][13.375, 9.0, 8.9375, 7.0625, 6.625][11.0, 7.8125, 6.9375, 5.9062, 5.25]'▁Dallas': [-inf, -inf, -inf, -inf, -inf][50.75, 17.125, 15.125, 10.5625, 10.3125][13.375, 9.0, 8.9375, 7.0625, 6.625][29.125, 22.0, 10.75, 6.4375, 5.2812][26.875, 7.5312, 5.8438, 1.1484, 1.0781]",


### Graph Pruning, Creation and Saving

#### Prune by Node/Edge Influence

We next inspect graph pruning process `prune_graph` following the transformation of our target logits raw target (logit) node attribution through the `compute_node_influence` and `compute_edge_influence` functions.

The first step to computing our node influences is to normalize our adjacency matrix above. We then use this normalized adjacency matrix to compute node influences influence via a numerical approach to Neumann series calculation.

Iterative computing of our node influence vector starts with seeding our `logit_weights` via our direct logit probabilities. We then multiply those weights with our adjacency matrix  `logit_weights @ A` to construct the initial `current_influence` vector and proceed to iteratively update this influence vector until it converges, multiplying increasing degrees of our adjacency matrix by the `current_influence`.
```python
    current_influence = logit_weights @ A
    influence = current_influence
    iterations = 0
    while current_influence.any():
        if iterations >= max_iter:
            raise RuntimeError(
                f"Influence computation failed to converge after {iterations} iterations"
            )
        current_influence = current_influence @ A
        influence += current_influence
        iterations += 1
```

##### Computing node influences



For this example, we first inspect the initial adjacency matrix post-normalization within the `compute_influence` function.

Note we are not updating the entire adjacency matrix in compute_influence, just a single axis of the matrix, an influence/current_influence vector (size n_total_nodes), so we won't see 2nd order nodes updated in A.

`logit_weights` starts off with as the targets axis of the adjacency matrix (dim 0) and then sets the last `n_logit` weights to the logit probabilities, all the other initial values are initially zero:
```python
    logit_weights = torch.zeros(
        graph.adjacency_matrix.shape[0], device=graph.adjacency_matrix.device
    )
    logit_weights[-n_logits:] = graph.logit_probabilities
```

As we multiply by the adjacency matrix in `compute_influence`, the first iteration will result in the last `n_logits` elements of the first column (the source weights of the first feature) contributing to the dot product with the logit probs, yielding the weighted sum of contributions of the first feature node to the logits (and so on for the remaining nodes).
See the validated/sampled calculation for the first iteration and first feature concretely:

Observe the first iteration consists of the weighted sum of the direct influences for each source feature on the logit probs:

**Analysis Point Data**: See below sampled data for the current attribution example at the beginning of the `Prune by Node Compute Influence`

In [22]:
analysis_injector.get_output("ap_node_compute_influence_init", skip=["trace_dict", "context", "iteration"])

Key,Inspected Output,Annotation
current_influence_shape,torch.Size([7390]),
normalized_tgt_logit_nodes,"[1.0, 1.0]","Verify normalization: `A[tta.graph_logit_indices, :].sum(dim=1)`"
nodeinf_normed_1st_order_idxs,"'▁Austin': [6394, 7378, 6889, 7361, 6466]'▁Dallas': [7378, 6394, 6889, 6466, 7131]",
nodeinf_normed_1st_order_vals,"'▁Austin': [0.0388, 0.0362, 0.0308, 0.0235, 0.0211]'▁Dallas': [0.0553, 0.0338, 0.0299, 0.0221, 0.0186]",
logit_weights,"[0.337891, 0.109863, 0.075684, 0.040527, 0.040527, 0.035645, 0.031494, 0.024536, 0.021606, 0.016846]",logit probabilites are injected as logit_weights to compute the initial influence vector
initial_influence_feat_0,3.3e-05,"Validate the initial element of our initial influence vector`logit_weights[-max_n_logits:].dot(A[-max_n_logits:, 0])`"
current_influence_feat_0,3.3e-05,The first element of our current influence vector `current_influence[0]` consists of the weighted sum of the direct influences for each source feature on the logit probs


See below the evolution of `current_influences` in the node influence computation for this example:

##### Neumann Series Convergence

In [23]:
trace_dict = analysis_injector["ap_node_compute_influence"]["trace_dict"]["node"]
stacked_trace = torch.stack(trace_dict)
fig = plot_ridgeline_convergence(
    data=stacked_trace, stats=None, title="Neumann Series Convergence Trace Ridgeline Plot"
)
fig.show()

Note you should see very little marginal distributional change after the first few iterations.
For most examples, you should see that after around the first few iterations, the distribution stabilizes, which suggests contributions from longer paths
are not significantly contributing to logit values.

- Iteration 0 (initial logit prob dot product) captures the weighted sum of the direct influences for each feature on each of the logit probs
- Iteration 1 captures for each feature, that feature's weighted influence on the iteration 0 direct logit prob influences (in other words, the weighted direct logit prob influences of features mediated one-hop through each feature)
- Iteration 2 the weighted one-hop prob influences of features mediated two-hop through each feature (i.e. the second iteration reflects the marginal influence of two-hop paths, multiplying the adjacency matrix feature influences by all the 1-hop current_influence vector elements)
- `current_influences` continues to get smaller and smaller as the order/path length increases!

In [24]:
# Nicely format top-k influence values and indices by iteration using orchestrator helper
convergence_iteration = analysis_injector["ap_node_compute_influence"]["iteration"]
print(f"Convergence occurred at iteration: {convergence_iteration}")

sample_tensor_output(stacked_trace, (0, 2, 4, 8), ["Iteration", "Top k Values", "Top k Indices"], 5, tablefmt="html")

Convergence occurred at iteration: 27


Iteration,Top k Values,Top k Indices
0,"[0.027714, 0.019121, 0.016848, 0.013618, 0.010803]","[7379, 7378, 6394, 6889, 7361]"
2,"[0.035103, 0.02459, 0.021745, 0.018485, 0.01488]","[7373, 7371, 7372, 7377, 7376]"
4,"[0.017208, 0.011942, 0.01104, 0.006215, 0.005568]","[7373, 7372, 7371, 7377, 7376]"
8,"[0.000125, 0.000107, 9.3e-05, 1.7e-05, 1.6e-05]","[7372, 7371, 7373, 7376, 7377]"


##### Computing `node_mask`

We then calculate a node_mask by finding a threshold (`find_threshold`) that meets our targeted explained influence using our `node_influence` and configured node_threshold.

```python
def find_threshold(scores: torch.Tensor, threshold: float):
    # Find score threshold that keeps the desired fraction of total influence
    sorted_scores = torch.sort(scores, descending=True).values
    cumulative_score = torch.cumsum(sorted_scores, dim=0) / torch.sum(sorted_scores)
    threshold_index = torch.searchsorted(cumulative_score, threshold)
    # make sure we don't go out of bounds (only really happens at threshold=1.0)
    threshold_index = min(threshold_index, len(cumulative_score) - 1)
    return sorted_scores[threshold_index]
```

Before we compute our `edge_mask`, our `node_mask` has filtered the number of non-zero src and target features to include only features that explain the specified `node_threshold` of influence

```python
pruned_matrix = graph.adjacency_matrix.clone()
pruned_matrix[~node_mask] = 0
pruned_matrix[:, ~node_mask] = 0
```

**Analysis Point Data**: See below sampled data for the current attribution example at the end of `Prune by Node Influence`.

In [25]:
analysis_injector.get_output("ap_graph_prune_node_influence_end")

Key,Inspected Output,Annotation
n_logits,10,
n_tokens,9,
node_mask_nonzero,1599,"Number of non-zero elements in the node mask, includes token and logit nodes plus minimum required nodes to meet specified node threshold"
node_threshold,0.8,
pruned_matrix_nonzero,769960,
node_mask_nonlogit_nontoken_nonzero,1580,"We always keep token and logit nodes, so we subtract them from the count"
percent_of_nodes_kept,21.38%,The percentage of nodes kept after pruning
percent_of_elements_kept,1.41%,The percentage of non-zero elements kept in the pruned adjacency matrix after applying our node_mask to both dimensions


##### Computing `edge_mask`

We can further reduce the graph size by filtering on the cumulative influence of all edges (0.98 threshold in this case).

- `compute_edge_influence` starts with a node-threshold-pruned adjacency matrix:
    ```python
    edge_scores = compute_edge_influence(pruned_matrix, logit_weights)
    ```
- The pruned_matrix is not initially normalized, but `compute_edge_influence` will normalize it prior to `compute_influence`
- After `computing_influence` (same fn/process as above) we see we have the feature influences vector for nodes that influence logits (weighted average of all collectively) above the given threshold
- Since we want to have our completed edge matrix to include the logit probs, we add in logit_weights to the `pruned_influence` vector before returning `edge_scores` as normalized_pruned matrix elementwise multiplied by the each element of the `pruned_influence` vector.
- We'll see below at the end we no longer have normalized target influence vectors but rather the sum of the src features will be the total influence that target feature had on the overall logits

- So our new `edge_scores` matrix will have rows where each target feature is the `pruned_influence` value for that feature multiplied by the `normalize_pruned` vector for each feature (from the `normalized_pruned` matrix which is the threshold-pruned adjacency matrix) 
- As we see below, this effectively scales our normalized feature scores for each target feature by our `pruned_influence` scores (sum of all paths above a threshold)
- By multiplying the normalized vector of influences for each target feature by every other source feature by the total influence of that target feature on the logits, **we scale for each target feature the source feature influences so they are weighted for that target feature's influence on logits by all paths**

- Because we scaled our normalized features by the pruned_influence values, *the sum of our target feature rows will now be the pruned_influence values*.

- You should observe below that the feature influences are quite distributed. In the case of the default example prompt, for the top logit, only 0.21 of the 0.34 logit probability (64%) is accounted for by the top 100 features, and the top feature only accounted for 0.0162 (4.8%) of the logit prob for our top logit.

**Analysis Point Data**: See below sampled data for the current attribution example at the end of `Compute Edge Influence`.

In [26]:
analysis_injector.get_output("ap_graph_prune_edge_influence_post_norm")

Key,Inspected Output,Annotation
edge_scores.shape,"(7390, 7390)","total_nodes, total_nodes"
normalized_pruned.shape,"(7390, 7390)","total_nodes, total_nodes"
pruned_influence.shape,"(7390,)",total_nodes
top_target_edge_vals,"'▁Austin': [0.0162, 0.0151, 0.0128, 0.0098, 0.0088]'▁Dallas': [0.0017, 0.001, 0.0009, 0.0007, 0.0006]",
top_target_edge_idxs,"'▁Austin': [6394, 7378, 6889, 7361, 6466]'▁Dallas': [7378, 6394, 6889, 6466, 7131]",
target_token_logit_prob,'▁Austin': 0.3379'▁Dallas': 0.0245,
top_1_edge_percent_logit_prob,'▁Austin': 0.0479'▁Dallas': 0.0693,
top_10_edges_percent_logit_prob,'▁Austin': 0.2841'▁Dallas': 0.2965,
top_100_edges_percent_logit_prob,'▁Austin': 0.6354'▁Dallas': 0.6535,
top_1000_edges_percent_logit_prob,'▁Austin': 0.9688'▁Dallas': 0.9739,


We then compute our `edge_mask` using the same `find_threshold` function as above to meet our `edge_threshold` target.
As we'll see, we're able to dramatically reduce the number of nonzero elements even while retaining a very large fraction of total edge influence.
Note below the sum of our edge_score attributions to target logits will equal the aggregate target logit probabilities.


In [27]:
analysis_injector.get_output("ap_graph_prune_edge_influence_pre_mask")

Key,Inspected Output,Annotation
edge_mask_numel,54612100,Number of elements in the edge mask
edge_mask_count_nonzero,400859,Number of non-zero elements in the edge mask
edge_mask_percent_nonzero,0.73%,Percentage of non-zero elements in the edge mask
node_mask_numel,7390,Number of elements in the node mask
node_mask_count_nonzero,1599,Number of non-zero elements in the node mask
node_mask_percent_nonzero,21.64%,Percentage of non-zero elements in the node mask
sum_edge_score_attribution_to_target_logits,0.735,"Sum of edge score attribution to target logits, `edge_scores[-n_logits:, :].sum()`"
sum_logit_probabilities,0.735,"Sum of logit probabilities for target logits, `logit_weights[-n_logits:].sum()`, should match aggregate edge score attribution"


##### Applying Edge Mask and Finalizing Score Matrix

- We next ensure proper graph connection by ensuring all feature and error nodes have outgoing edges and all feature nodes have incoming edges

```python
    old_node_mask = node_mask.clone()
    # Ensure feature and error nodes have outgoing edges
    node_mask[: -n_logits - n_tokens] &= edge_mask[:, : -n_logits - n_tokens].any(0)
    # Ensure feature nodes have incoming edges
    node_mask[:n_features] &= edge_mask[:n_features].any(1)
```

- `node_mask[: -n_logits - n_tokens] &= edge_mask[:, : -n_logits - n_tokens].any(0)`
  - **Slices** `node_mask` to exclude the last `n_logits + n_tokens` elements.
  - Uses `&=` (in-place logical AND) to update the mask.
  - `edge_mask[:, : -n_logits - n_tokens]` selects columns corresponding to the same nodes.
  - `.any(0)` checks if **any edge exists** for each node (across all rows dim=0), returning a boolean tensor.
  - The mask is updated so that only nodes with at least one outgoing edge remain `True`. (source nodes (columns) that are not currently masked and do not have any outgoing edges to target nodes will now be masked)
- For the default example prompt, concretely, we examine the 7349 source influence columns in this case and return whether any target node (outgoing edge) exists

  ```python
  node_mask.count_nonzero()
  tensor(1584, device='cuda:0')
  edge_mask[:, : -n_logits - n_tokens].shape
  torch.Size([7368, 7349])
  edge_mask[:, : -n_logits - n_tokens].any(0).shape
  torch.Size([7349])
  ```

- `node_mask[:n_features] &= edge_mask[:n_features].any(1)`
    - Slices `node_mask` to the first `n_features` elements.
    - `edge_mask[:n_features]` selects rows for feature nodes.
    - `.any(1)` checks if **any incoming edge exists** for each feature node (across all columns).
    - Updates the mask so that only feature nodes with at least one incoming edge remain `True`.

- We prune iteratively all nodes that are missing incoming or outgoing edges (in this case, no further pruning was necessary because our pruned node_mask already equaled our original mask, meaning our prune operations didn't make any changes)
    ```python
    torch.all(node_mask == old_node_mask)
    ```
- We finally calculate the cumulative influence scores, sorting by `node_influence` descending and calculating the cumulative scores as a fraction of the total sorted scores and return our calculated node_mask, `edge_mask` and final_scores as a `PruneResult`
```python
    # Calculate cumulative influence scores
    sorted_scores, sorted_indices = torch.sort(node_influence, descending=True)
    cumulative_scores = torch.cumsum(sorted_scores, dim=0) / torch.sum(sorted_scores)
    final_scores = torch.zeros_like(node_influence)
    final_scores[sorted_indices] = cumulative_scores
```

- Be aware that pytorch will round representations to 4 digits by default, but many of our non-zero contributing influences will appear to be 0 with this granularity.
- In the case of the default example:
    - 7089 of our 7368 nodes contribute to the cumulative total score (279 zero scores in cumulative_scores)
    - The top 30 feature influences account for 34% of our aggregate logit probability.

**Analysis Point Data**: See below sampled data for the current attribution example at the end of `Applying Edge Mask and Finalizing Score Matrix`.

In [28]:
analysis_injector.get_output("ap_graph_prune_edge_influence_end")

Key,Inspected Output,Annotation
cumulative_scores_first_10,"[0.057, 0.099, 0.135, 0.163, 0.191, 0.213, 0.235, 0.25, 0.26, 0.269]",First 10 values of cumulative_scores
cumulative_scores_last_10,"[1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]",Last 10 values of cumulative_scores
cumulative_scores_30th_feature,0.34,Cumulative score of the 30th feature
contributors_mask_sum,7140,Number of non-zero contributors in cumulative_scores (< 1)
sorted_indices_first_10,"[7373, 7371, 7372, 7378, 7377, 7376, 7379, 7374, 7375, 6394]",First 10 values of sorted_indices
sorted_scores_first_10,"[0.116, 0.086, 0.075, 0.058, 0.056, 0.046, 0.045, 0.029, 0.021, 0.02]",First 10 values of sorted_scores (from node_influence)
final_scores_first_10,"[0.747, 0.653, 0.892, 0.621, 0.641, 0.68, 0.988, 0.363, 0.722, 0.66]",The returned final_scores reflects the CUMULATIVE node influences of each feature up to that feature number (create_nodes requires the cumulative_scores form of features for graph construction)
node_mask_count_nonzero,1599,Number of non-zero elements in node_mask
node_mask_sparsity,21.64%,Final sparsity of node_mask (percentage of non-zero elements)
edge_mask_count_nonzero,400859,Number of non-zero elements in edge_mask


##### Graph Creation and Saving

- We then use our node_mask, edge_mask and cumulative_scores to construct our graph:
    ```python
        tokenizer = AutoTokenizer.from_pretrained(graph.cfg.tokenizer_name)
        nodes = create_nodes(graph, node_mask, tokenizer, cumulative_scores, scan)
        used_nodes, used_edges = create_used_nodes_and_edges(graph, nodes, edge_mask)
        model = build_model(graph, used_nodes, used_edges, slug, scan, node_threshold, tokenizer)

        # Write the output locally
        with open(os.path.join(output_path, f"{slug}.json"), "w") as f:
            f.write(model.model_dump_json(indent=2))
        add_graph_metadata(model.metadata.model_dump(), output_path)
        logger.info(f"Graph data written to {output_path}")

        total_time_ms = (time.time() - total_start_time) * 1000
        logger.info(f"Total execution time: {total_time_ms=:.2f} ms")
    ```
- When the graph is initially constructed, the `graph.active_features` tensor is created from this mapping in the Graph instantiation:
    ```python
    ...
    active_features=activation_matrix.indices().T,
    activation_values=activation_matrix.values(),
    ...
    adjacency_matrix=full_edge_matrix,
    ```
- defined by:
    ```python
        active_features (torch.Tensor): A tensor of shape (n_active_features, 3)
            containing the indices (layer, pos, feature_idx) of the non-zero features
            of the model on the given input string.
        adjacency_matrix (torch.Tensor): The adjacency matrix. Organized as
            [active_features, error_nodes, embed_nodes, logit_nodes], where there are
            model.cfg.n_layers * len(input_tokens) error nodes, len(input_tokens) embed
            nodes, len(logit_tokens) logit nodes. The rows represent target nodes, while
            columns represent source nodes.
    ```

### Cleanly teardown analysis injection

In [29]:
# Teardown hooks after graph generation
if enable_analysis_injection:
    print("\nDisabling analysis hooks...")
    try:
        analysis_injector.teardown()
        print("✓ Analysis injector cleaning torn down.")
        if log_path := getattr(analysis_injector, "analysis_log", None):
            print(f"Analysis log available for inspection: {log_path}")
    except Exception as e:
        print(f"Error while tearing down analysis injector: {e}")


Disabling analysis hooks...
✓ Analysis injector cleaning torn down.
Analysis log available for inspection: /tmp/attribution_flow_analysis_20261103_142114.log


### Saving and Visualizing Attribution Graphs

In this section, we'll demonstrate how to save the generated attribution graphs and prepare them for visualization. The CircuitTracerAdapter integrates with Interpretune's AnalysisStore to persistently store graph data.

In [30]:
import os
import socket

from circuit_tracer.frontend.local_server import serve


enable_iframe = False  # whether to enable the IFrame display or not

port = 8046
server = serve(data_dir=ct_module.circuit_tracer_cfg.graph_output_dir, port=port)
port_forwarding = False  # whether to use port forwarding or not
# Host used to build the graph URL. Defaults to this machine's hostname, which is what you want when
# viewing from another machine on the same network. Override with IT_GRAPH_SERVER_HOST, or set
# port_forwarding=True above to use localhost (e.g. over an SSH tunnel).
hostname = os.environ.get("IT_GRAPH_SERVER_HOST") or socket.gethostname()

if port_forwarding:
    hostname = "localhost"  # use localhost for port forwarding
    print(
        f"Using port forwarding (ensure it is configured) and localhost. "
        f"Open your graph here at http://{hostname}:{port}/index.html"
    )
else:
    print(
        f"Not using port forwarding. Use the IFrame below, or open your graph here "
        f"directly at http://{hostname}:{port}/index.html"
    )

if enable_iframe:
    from IPython.display import IFrame

    # Display the IFrame with the graph visualization
    print(f"Displaying graph visualization in IFrame at http://{hostname}:{port}/index.html")
    display(IFrame(src=f"http://{hostname}:{port}/index.html", width="100%", height="800px"))

Not using port forwarding. Use the IFrame below, or open your graph here directly at http://speediedl:8046/index.html


In [31]:
server.stop()

### Next Steps and Future Extensions

This notebook demonstrates the basic scaffolding for the CircuitTracerAdapter. The current implementation provides:

1. **Basic Integration**: CircuitTracerAdapter integrates with Interpretune's session management
2. **Configuration**: CircuitTracerConfig allows customization of attribution parameters
3. **Protocol Support**: CircuitAnalysisBatchProtocol defines the interface for batch processing
4. **Adapter Composition**: Seamless integration with other Interpretune adapters

#### Future Extensions:

1. **Full Implementation**: Complete the adapter methods to actually generate attribution graphs
2. **Batch Processing**: Support for efficient batch attribution analysis
3. **Advanced Analysis**: Integration with AnalysisOp for complex circuit analysis workflows
4. **Visualization**: Built-in support for graph visualization and exploration
5. **Caching**: Intelligent caching of attribution results for faster iteration
6. **Model Support**: Extended support for different model architectures beyond GPT-2

#### Resources:

- [Circuit Tracer Documentation](https://github.com/jacobdunefsky/circuit-tracer)
- [Interpretune Documentation](https://github.com/speediedan/interpretune)
- [Attribution Methods Paper](https://arxiv.org/abs/2310.10348)

This scaffold provides a solid foundation for building sophisticated circuit analysis workflows with Interpretune and Circuit Tracer.